# <h1><center>Gravitational-Wave Explorer: A Beginner's Guide–Notebook 2 </center><h1>


#Notebook 2 of 3

Authors: Bradlee Tejeda$^{1}$, Rachel Langgin$^{2}$

$^{1}$Las Vegas Academy of the Arts

$^{2}$University of Nevada, Las Vegas

# Notebook 2: Generating Waveforms

At the end of Notebook 1, you saw how orbiting black holes disturb spacetime and produce gravitational waves. You also asked an important question:

> *How do we measure something that stretches and squeezes space itself?*

This notebook answers that question.

We now move from understanding the physics of gravitational waves to analyzing real signals.

## What You'll Learn:

* How detectors like LIGO and Virgo measure incredibly small distortions in spacetime

* Access open data from a real gravitational-wave event, including the first binary black hole merger observed, GW150914, and the first binary neutron star coalescence, GW170817

* Generate theoretical waveforms that model colliding compact object coalescences

* Compare simulated signals to observed data

In Notebook 1, you studied mass, motion, curvature, and spacetime. In this notebook, you will see how those physical ideas appear inside real data.

We begin by importing the scientific Python tools that allow us to load data, generate waveforms, and visualize signals. As you read through the import cell, pay attention to what each library allows us to do — these are the same kinds of tools professional gravitational-wave scientists use.

⚠️ Warning: restart the runtime after running the cell below. ⚠️

To do so, click "Runtime" in the menu and choose "Restart and run all".

#Part 1: GW Detectors

In [ ]:
# === Plotting ===
import matplotlib.pyplot as plt  # Import the popular plotting library for creating graphs and visualizations.
from PIL import Image as PILImage # Import PIL's Image class for image processing tasks (renamed to avoid conflicts).

# === Colab utilities ===
import os  # Provides a way to use operating system functionality like file paths.
import numpy as np  # Fundamental package for numerical computations, especially arrays and matrices.
from google.colab import files, drive  # Colab helpers to upload/download files and mount Google Drive.

# -- For Google Colab
!pip install --quiet "gwpy==3.0.12" "astropy==5.3.4" matplotlib==3.7.3 gwosc # Installs specific versions of gravitational wave analysis packages in the notebook environment.

## Section 1: Gravitational Wave (GW) Detectors

## GW Detectors Around the World

The image below shows the current and planned locations of gravitational wave detectors around the world. These observatories work together to detect and verify the incredibly faint ripples in spacetime caused by massive events in the universe. Below are descriptions of the major GW detectors:

- **LIGO Hanford (Washington, USA)**: One of the two LIGO observatories in the United States. It played a key role in the first direct detection of gravitational waves (GW150914).
- **LIGO Livingston (Louisiana, USA)**: The second LIGO site in the U.S., which works in tandem with LIGO Hanford for improved accuracy and signal triangulation.
- **Virgo (Pisa, Italy)**: A European interferometer that joined LIGO-Virgo in 2017. It helped with further refining sky localization precision.
- **KAGRA (Kamioka Mine, Japan)**: The first GW detector built underground and cooled to cryogenic temperatures, improving sensitivity to lower frequency signals.
- **GEO600 (Hanover, Germany)**: A smaller detector used primarily for testing new technologies that will be used in larger detectors. It is also "always on" in case of a nearby Supernova (stellar explosion) observation.
- **LIGO India (under construction)**: Planned and currently under construction. An additional detector around the globe will further increase sensitivity and build a network of interferometers capable of localizing GW events by adding another geographically distant detector.

In [ ]:
import requests  # For sending HTTP requests to get the image from the web.
from io import BytesIO  # For handling byte streams, letting us open images from downloaded content.

# Load the image directly from the URL
url = "https://www.ligo.caltech.edu/system/avm_image_sqls/binaries/136/huge/GW_Detector_Map_v5.jpg?1608063350"  # Link to the GW detector map image.
response = requests.get(url)  # Make an HTTP GET request to fetch the image data.
img = PILImage.open(BytesIO(response.content))  # Open the image using PIL from the downloaded byte content.

# Display the image
fig, ax = plt.subplots(figsize=(10, 6))  # Create a figure and axes with specified size for better visibility.
ax.imshow(img)  # Show the image on the axes.
ax.axis('off')  # Hide the axis ticks and labels for a cleaner look.
plt.show()  # Render the image in the output cell.

##Section 2: Key Observations – Binary Black Holes vs. Neutron Star Mergers

## What Have We Observed?

Since the first detection in 2015, detectors such as LIGO, Virgo, and KAGRA have observed dozens of compact binary coalescences (CBCs) — mergers of extremely dense objects.

The majority of detected events are binary black hole (BBH) mergers. Fewer involve neutron stars. A neutron star is an extremely dense stellar remnant formed when a massive star explodes, leaving behind a city-sized object made almost entirely of tightly packed neutrons.

These detections fall into three main categories:

### 1. **Binary Black Hole (BBH) Mergers**:

* First observed event: GW150914

* Two black holes spiral together and merge.

* These events produce strong gravitational-wave signals.

* No light is detected, because black holes do not emit electromagnetic radiation.

BBH mergers make up most of the events observed so far.

### 2. **Binary Neutron Star (BNS) Mergers**:
* First observed event: GW170817

* Two neutron stars collide. This event was detected in both gravitational waves and light (gamma rays, optical, radio).

* It confirmed that neutron star mergers produce heavy elements such as gold and platinum.

These events are rarer than BBH mergers but scientifically rich because they can be observed across multiple types of signals.

### 3. **Black Hole Neutron Star (BHNS) Mergers**:
* First observed events: GW200105 and GW200115

* A black hole merges with a neutron star.

* In the first confirmed detections, no light was observed, likely because the neutron star was swallowed before being torn apart.

These events confirm that mixed systems exist in nature.

### Each merger type produces a slightly different GW signal. By studying the waveform — its frequency, amplitude, and duration – we can infer the system's properties and classify which objects created the GW.


In [ ]:
from IPython.display import display, HTML, Image as IPyImage, Markdown  # Imports for rich display in notebooks: allows showing Markdown, HTML, images.
from matplotlib.widgets import Slider  # Import for creating sliders in matplotlib (optional here but good for 2D/3D plots with sliders).
from ipywidgets import FloatSlider # Import for creating sliders in ipywidgets
from mpl_toolkits.mplot3d import Axes3D  # Enables 3D plotting support in matplotlib.

# === Jupyter/Colab UI ===
import ipywidgets as widgets  # Widgets for interactive controls like sliders, dropdowns, buttons.
from ipywidgets import interact  # Easy function for making simple interactive UIs.

# ---------------------------------------------------------------
# Event details dictionary
# ---------------------------------------------------------------
# This dictionary stores information about gravitational-wave events.
# Each event has a short description in Markdown and a URL to an illustrative image.
cbc_events = {
    "GW150914": {
        "desc": (
            "**GW150914**\n\n"
            "First-ever detection of gravitational waves on September 14, 2015. "
            "Two black holes (36 and 29 solar masses) merged about 1.3 billion light-years away. "
            "This event confirmed a major prediction of Einstein’s general theory of relativity."
            "For more information, check out LIGO's [detection page on GW150914](https://ligo.org/detections/gw150914/)."
        ),
        "video_url": "https://www.youtube.com/embed/I_88S8DWbcU"
    },
    "GW170817": {
        "desc": (
            "**GW170817**\n\n"
            "Detected on August 17, 2017. First neutron star merger observed in both gravitational waves and light. "
            "This marked the beginning of multi-messenger astronomy."
            "For more information, check out [LIGO's detection page on GW170817](https://ligo.org/detections/gw170817/)."
        ),
        "video_url": "https://www.youtube.com/embed/V6cm-0bwJ98"
    }
}

# ---------------------------------------------------------------
# Interactive function
# ---------------------------------------------------------------
# This function takes an event name (e.g., "GW150914"), then:
# - Displays its Markdown-formatted description.
# - Loads and shows the associated video from its URL.
def show_cbc_info(event):
    display(Markdown(cbc_events[event]['desc']))

    # Extract video ID from embed URL
    video_id = cbc_events[event]['video_url'].split("/")[-1]

    # YouTube thumbnail image URL
    thumbnail_url = f"https://img.youtube.com/vi/{video_id}/hqdefault.jpg"

    # Clickable image linking to YouTube watch page
    watch_url = f"https://www.youtube.com/watch?v={video_id}"

    html_code = f"""
    <a href="{watch_url}" target="_blank">
        <img src="{thumbnail_url}" width="560">
    </a>
    <p>Click the video thumbnail to watch the merger simulation videos in a new tab.</p>
    """

    display(HTML(html_code))

# ---------------------------------------------------------------
# Interact widget
# ---------------------------------------------------------------
# This line creates an interactive dropdown widget automatically,
# letting the user select which event to display.
# When the selection changes, show_cbc_info runs automatically to update the output.
interact(show_cbc_info, event=list(cbc_events.keys()));

In [ ]:
from IPython.display import Audio  # Imports for displaying audio players

# ===========================
# Gravitational Wave Audio Clips with Explanations
# ===========================
# This dictionary stores metadata about different gravitational wave audio clips.
# Each key is an event+detector name (e.g., "GW150914_H1").
# Each entry includes:
#  - 'url': link to the processed audio file
#  - 'desc': a Markdown-formatted description for educational context
gw_audio_clips = {
    "GW150914_H1": {
        "url": "https://raw.githubusercontent.com/ii-vnx/gw-audio-clips/main/GW150914_H1.wav",
        "desc": (
            "## GW150914 - Hanford Detector (H1)\n"
            "**Recorded at:** LIGO Hanford Observatory, Washington, USA\n\n"
            "This is real detector data for the first-ever directly detected gravitational wave event on **September 14, 2015**. "
            "The two black holes spiraled into each other over 1.3 billion light-years away. "
            "The data has been adjusted so human ears can hear the 'chirp' as the black holes merge."
        )
    },

    "GW150914_L1": {
        "url": "https://raw.githubusercontent.com/ii-vnx/gw-audio-clips/main/GW150914_L1.wav",
        "desc": (
            "## GW150914 - Livingston Detector (L1)\n"
            "**Recorded at:** LIGO Livingston Observatory, Louisiana, USA\n\n"
            "This is the Livingston detector’s version of the GW150914 signal. "
            "The data has been adjusted so human ears can hear the 'chirp' as the black holes merge."
            "Comparing Hanford and Livingston data helps confirm the event and triangulate its source."
        )
    },

    "GW170817_H1": {
        "url": "https://raw.githubusercontent.com/ii-vnx/gw-audio-clips/main/GW170817_H1.wav",
        "desc": (
            "## GW170817 - Hanford Detector (H1)\n"
            "**Recorded at:** LIGO Hanford Observatory, Washington, USA\n\n"
            "Detected on **August 17, 2017**, this was the first neutron star merger ever observed in both gravitational waves **and** light (gamma rays, visible light, etc.). "
            "The data has been adjusted so human ears can hear the 'chirp' as the black holes merge."
            "It represents the moment when two neutron stars collided, creating heavy elements like gold and platinum."
        )
    },

    "GW170817_L1": {
        "url": "https://raw.githubusercontent.com/ii-vnx/gw-audio-clips/main/GW170817_L1.wav",
        "desc": (
            "## GW170817 - Livingston Detector (L1)\n"
            "**Recorded at:** LIGO Livingston Observatory, Louisiana, USA\n\n"
            "This is the same event (GW170817) as seen by Livingston. "
            "The data has been adjusted so human ears can hear the 'chirp' as the black holes merge."
            "Multiple detectors ensure we can confidently identify the signal and localize the source."
        )
    },

    "GW170817_V1": {
        "url": "https://raw.githubusercontent.com/ii-vnx/gw-audio-clips/main/GW170817_V1.wav",
        "desc": (
            "## GW170817 - Virgo Detector (V1)\n"
            "**Recorded at:** Virgo Observatory, Pisa, Italy\n\n"
            "Virgo joined the global network in 2017, just in time to observe GW170817. "
            "This clip is real adjusted Virgo detector data of the neutron star merger. "
            "Including Virgo improved localization dramatically, proving the power of a global gravitational wave detector network."
        )
    },
}

# ---------------------------------------------------------------
# Dropdown widget
# ---------------------------------------------------------------
# Here we create an interactive dropdown using ipywidgets.Dropdown.
# The dropdown shows the title of each clip for students to choose from.
# It uses a list of tuples: (Display Label, Key in gw_audio_clips)
# This makes the UI friendly while keeping the backend references consistent.
dropdown = widgets.Dropdown(
    options=[(value["desc"].split('\n')[0][3:], key) for key, value in gw_audio_clips.items()],
    description='Choose a Recording:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='70%')
)

# ---------------------------------------------------------------
# Display function
# ---------------------------------------------------------------
# This function is called when the user picks an item from the dropdown.
# It does two things:
#  - Displays the Markdown description with rich formatting
#  - Embeds and plays the audio clip directly in the notebook
def play_audio(selection):
    clip = gw_audio_clips[selection]
    display(Markdown(clip['desc']))  # Show the event's educational description
    display(Audio(url=clip['url']))  # Render the audio player so students can listen

# ---------------------------------------------------------------
# Interact
# ---------------------------------------------------------------
# Finally, we use widgets.interact to bind our play_audio function to the dropdown.
# When students choose a recording from the dropdown, play_audio runs automatically.
# This turns the notebook into an interactive audio exploration tool!
widgets.interact(play_audio, selection=dropdown);

<h3>Quiz Question:<h3>

**Which Gravitational-Wave signal lasted longer in the detector?**

- A) GW150914 (Binary Black Hole)
- B) GW170817 (Binary Neutron Star)

<details>

<summary> Click to reveal the correct answer and explanation</summary>

B) GW170817 (Binary Neutron Star)

Explanation:

The binary neutron star signal lasted much longer in the detector because neutron stars are much less massive than black holes. Lower-mass systems orbit each other more slowly and spend more time spiraling inward before they merge. As a result, the Gravitational Wave “chirp” builds up gradually over tens of seconds.

In contrast, the black holes in GW150914 were much more massive, so they merged quickly. Their signal lasted only a fraction of a second in the LIGO detector.
</details>

##Section 3: Next-Generation Detectors (XG)

Current detectors like LIGO and Virgo have already detected dozens of compact binary mergers. However, they are limited in how far into the universe they can “see.”

Next-generation (XG) detectors are being designed to dramatically improve sensitivity, allowing us to detect weaker and more distant signals.

## Next-Generation GW Detectors

### Einstein Telescope (ET)
* A proposed underground gravitational-wave detector in Europe.

* Designed in a triangular configuration with 10 km arms.

* Built underground to reduce seismic and environmental noise.

* Expected to be about 10 times more sensitive than current detectors.

* Could detect black hole mergers from much earlier in cosmic history, possibly when the first stars were forming.

$\small$ ![Einstein Telescope](https://www.aei.mpg.de/535047/original-1740491299.webp?t=eyJ3aWR0aCI6OTYwLCJmaWxlX2V4dGVuc2lvbiI6IndlYnAiLCJvYmpfaWQiOjUzNTA0N30%3D--dc4b3de4fa8a6220887b8faeda672285208676c8) $\small$

### Cosmic Explorer (CE)
* A proposed next-generation U.S. detector.

* Would have arms about 40 km long (compared to LIGO’s 4 km arms).

* Longer arms mean a larger measurable change in length when a gravitational wave passes, increasing precision.

* Also expected to improve sensitivity by roughly a factor of 10.

![Cosmic Explorer](https://news.syr.edu/wp-content/uploads/2021/11/Cosmic-400x300-1.jpg)

### Why Does Sensitivity Matter?

Gravitational wave strength decreases with distance. If a detector is 10 times more sensitive in strain, it can detect sources about 10 times farther away. Since volume scales as distance$^3$, that means observing roughly 1,000 times more events.

With next-generation detectors, scientists expect to:

* Detect most stellar-mass black hole mergers occurring in the observable universe

* Observe binary neutron star mergers at much greater distances

* Study how black holes formed and evolved over cosmic time

* Potentially detect new or rare types of sources

Rather than simply hearing occasional nearby events, next-generation detectors will allow GW astronomy to become a precision population science.

![Next-Gen Visible Universe](https://upload.wikimedia.org/wikipedia/commons/9/9e/Schematic_diagram_of_the_history_of_the_Universe.jpg)

<h3>Quiz Question<h3>

**What is the difference between current generation detectors like LIGO and Virgo, and the next-generation detectors like CE and ET?**

[Fill in Student Answer here]

<details>

<summary> Click to reveal the correct answer and explanation</summary>

Current detectors like LIGO and Virgo can detect gravitational waves from massive events, but only out to a certain distance and sensitivity.

Next-generation detectors like Cosmic Explorer (CE) and Einstein Telescope (ET) are being designed to be much more sensitive. This means they will:

- Detect signals that are much weaker

- Observe events from much farther away in the universe

- See signals for a longer amount of time before merger

In short, next-generation detectors will allow us to see more events, from earlier in cosmic history, and measure them more precisely.

</details>



---



# Part 2: How Does LIGO Detect a Gravitational Wave in Real Time?

LIGO measures the tiniest changes in distance — thousands of times smaller than a proton — using laser interferometry. When a gravitational wave passes through Earth, it stretches space in one direction and compresses it in another, causing one LIGO arm to grow slightly while the other shrinks. This change creates an interference pattern in the laser light, allowing us to “see” the ripple from events such as black hole or neutron star mergers in real time.

Thanks to its global network and advanced computing pipelines, LIGO can detect and announce a GW event within seconds, often triggering both space- and ground-based telescopes worldwide.

> Watch the animation below to see how LIGO’s interferometers detect these tiny distortions in real space:

In [ ]:
display(HTML('<iframe width="560" height="315" src="https://www.youtube.com/embed/tQ_teIUb3tE?si=rTnDyFoJ2UMnDAEb" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>'))


#What is Strain?

Strain is the change of relative separation of the test particles in the arms of the detectors, over their original undisturbed distance:

<center> <big> h = $\frac{\Delta L}{L}$</center> <big>

Where:

*   $\Delta L$ is the change in length caused by the GW,
*   L is the original length of the interferometer arm.

TThe detection of this fractional change allows GWs to be observed. Strain is usually incredibly small (on the order of $10^{-21}$) — but detectable thanks to LIGO’s extreme sensitivity.

#What is Chirp Mass?

The chirp mass ($\mathscr{M}$) is a special combination of the two masses in a binary system. It is important because it determines how the gravitational wave changes as the objects spiral together.

The formula is:

<center>$$\mathscr{M} = \frac{(m_1 m_2)^{3/5}}{(m_1 + m_2)^{1/5}}$$</center>

Here:

* $m_1$ and $m_2$ are the masses of the two objects (for example, two black holes or neutron stars).

The chirp mass tells us:

* How fast the orbit shrinks: bigger chirp mass means objects spiral together faster.

* How quickly the gravitational wave frequency rises: the “pitch” of the wave increases faster for larger chirp mass. More massive objects orbit and collide faster than less massive objects.

* How strong the signal is: a high chirp mass usually produces a louder/stronger gravitational wave signal.

You can think of it as the "main factor" controlling the pattern of the signal we detect. By combining the chirp mass with the mass ratio ($q = m_2 / m_1$), scientists can determine the individual masses of the two objects. So the chirp mass sets the main pattern of the signal, while the mass ratio fine-tunes the details

Let's plot the observed data from the binary neutron star merger, GW170817.
Neutron stars merge more slowly than black holes because their smaller mass produces weaker gravity, stretching out the signal so it is easier to observe for a longer period of time.

In [ ]:
# Import the function that retrieves the GPS timestamp of a specific gravitational wave event
from gwosc.datasets import event_gps

# Import TimeSeries from GWPy to fetch and manipulate gravitational wave strain data
from gwpy.timeseries import TimeSeries

#### Plot from Notebook 2 is posted here:

# Enable inline plotting so that graphs display directly below each code cell
%matplotlib inline

# Get the GPS time (in seconds) of the GW170817 event (a neutron star merger)
gps_time = event_gps("GW170817")

# Define the time window: 15 seconds before and 15 seconds after the event
start = int(gps_time) - 15
end = int(gps_time) + 15

# Fetch the open strain data from the LIGO Livingston detector (L1) during the defined time window
# 'verbose=True' allows us to see download status and diagnostic output
hdata = TimeSeries.fetch_open_data("L1", start, end, verbose=True)

# Plot the strain data as a function of time
hdata.plot()

# Add a title to the time-domain plot
plt.title("LIGO Livingston Strain Data Around GW170817")

# Label the x-axis as GPS time (in seconds)
plt.xlabel("GPS Time [s]")

# Label the y-axis as strain (dimensionless quantity)
plt.ylabel("Strain")

# Add a grid to the plot to make it easier to read
plt.grid(True)

# Display the time-domain plot
plt.show()

# Compute the Amplitude Spectral Density (ASD) from the strain data
# This shows the signal strength as a function of frequency
asd = hdata.asd(fftlength=4)  # Use a 4-second FFT length to average over the data

# Create a new figure for the frequency-domain (ASD) plot
plt.figure(figsize=(10, 5))

# Plot the ASD curve
asd.plot()

# Add a title to the ASD plot
plt.title("GW170817: Frequency Domain (Amplitude Spectral Density)")

# Label the x-axis as frequency in hertz (Hz)
plt.xlabel("Frequency [Hz]")

# Label the y-axis as ASD in strain per root hertz
plt.ylabel(r"ASD [strain/$\sqrt{\mathrm{Hz}}$]")

# Add a grid to help interpret the frequency plot
plt.grid(True)

# Adjust layout to ensure labels and title fit cleanly
plt.tight_layout()

# Display the frequency-domain plot
plt.show()

# Explanation of the Graphs for GW170817

## Time Domain – Gravitational-Wave Signal

What you are seeing:

- The strain measured by the LIGO Livingston detector. Strain is the fractional stretching and squeezing of spacetime caused by a passing gravitational wave.
- The signal comes from the binary neutron star merger GW170817.
- The oscillations become faster and larger in amplitude near the end. This increasing frequency and amplitude is called a “chirp.”

Why this happens:

As the neutron stars lose energy through gravitational radiation, their orbit shrinks. A smaller orbit means higher orbital speed, which produces higher-frequency and stronger gravitational waves just before merger.

Key terms:

- **Strain**: Fractional change in length caused by a gravitational wave.
- **Time domain**: Viewing the signal as it changes over time.

---

## Frequency Domain – Amplitude Spectral Density

What you are seeing:

- A breakdown of the signal’s strength at different frequencies.
- GW170817’s strongest signal appears near ~100 Hz, where LIGO is most sensitive.
- Lower frequencies are more affected by detector noise.

Key terms:

- **Amplitude Spectral Density (ASD)**: Signal strength per $\sqrt{Hz}$, used to compare signal to detector noise.
- **Frequency domain**: Viewing the signal in terms of its frequency content rather than time evolution.

<h3>Quiz Question:</h3>

<b>During the observation of GW170817, scientists noted a gradual increase in both the frequency and amplitude of the gravitational wave signal leading up to the merger. Which of the following best explains this behavior, and what key parameter primarily determines how the signal evolves in time?</b>

<ul>
    <li>A) As the neutron stars orbit closer, they emit less gravitational radiation, causing the signal to fade. The key parameter is the orbital angular momentum.</li>
    <li>B) The stars' rotation slows down over time, reducing their gravitational influence. The key parameter is the orbit radius.</li>
    <li>C) The binary system loses energy via gravitational wave emission, shrinking the orbit and increasing orbital speed. The <b>chirp mass</b> governs the evolution of the waveform’s frequency and amplitude.</li>
    <li>D) The neutron stars increase in mass as they spiral in, boosting the gravitational wave signal. The main factor is the gravity from nearby stars.</li>
</ul>

<details>
<summary> Click to reveal the correct answer and explanation</summary>
  <p><b>Correct Answer: C</b></p>
  <p>The binary system emits Gravitational Waves, which carry energy away and shrink the orbit. This causes an increase in both frequency and amplitude — the "chirp." The <b>chirp mass</b> governs this evolution.</p>



---



#Part 3: Waveforms

In [ ]:
!pip install bilby --quiet
!pip install pycbc --quiet

In [ ]:
# More GW Libraries
import bilby  # GW modeling and inference.
from pycbc.waveform import get_td_waveform  # Time-domain waveform generator.
from pycbc.types import TimeSeries as PTS  # Time-series data handling.
import lalsimulation  # Detailed waveform models.
import astropy.constants as const  # Physical constants.

bilby.gw.source.lal_binary_black_hole  # Bilby BBH waveform function.

#Simulate Your Own Gravitational Waveform – Frequency Domain

Ever wondered how spacetime would stretch and squeeze if two black holes merged right next to Earth? Now, you’re the scientist. Using real physics engines (LALSuite + Bilby), you’ll simulate and visualize a black hole merger and explore how changing parameters like mass, spin, or distance changes the resulting gravitational wave signal.


Step 1: Set Your Black Hole Parameters

Use sliders to choose the mass and spin of the two black holes. The waveform will update automatically based on your values.

In [ ]:
# Suppress Bilby's INFO and WARNING logs for cleaner notebook output
bilby.core.utils.logger.setLevel("ERROR")

# ---------------------------------------------------------------
# STEP 1: Define default Injection Parameters
# ---------------------------------------------------------------
# These parameters describe the physical properties of the binary black hole system we simulate.
default_params = dict(
    mass_1=35.0,            # Mass of the primary black hole (in solar masses)
    mass_2=30.0,            # Mass of the secondary black hole (in solar masses)
    a_1=0.4,                # Dimensionless spin magnitude of the primary BH
    a_2=0.3,                # Dimensionless spin magnitude of the secondary BH
    tilt_1=0.0,             # Tilt angle of the primary spin (radians)
    tilt_2=0.0,             # Tilt angle of the secondary spin (radians)
    phi_12=0.0,             # Azimuthal angle difference between spins
    phi_jl=0.0,             # Angle between total angular momentum and orbital angular momentum
    luminosity_distance=1000.0, # Distance to the source in megaparsecs (Mpc)
    theta_jn=0.4,           # Inclination angle between total angular momentum and line of sight
    psi=2.659,              # Polarization angle
    phase=1.3,              # Phase at coalescence
    geocent_time=1126259642.413, # GPS time at geocenter
    ra=1.375,               # Right Ascension (celestial coordinate)
    dec=-1.2108             # Declination (celestial coordinate)
)

# ---------------------------------------------------------------
# STEP 2: Define waveform generation arguments
# ---------------------------------------------------------------
# Specify waveform model and frequency parameters
waveform_arguments = dict(
    waveform_approximant='IMRPhenomPv2', # Phenomenological waveform model including precession
    reference_frequency=50.0,            # Reference frequency for phase calculations (Hz)
    minimum_frequency=20.0               # Minimum frequency of the waveform (Hz)
)

# ---------------------------------------------------------------
# STEP 3: Set up the waveform generator object
# ---------------------------------------------------------------
# Generates frequency-domain gravitational wave strain given parameters
waveform_generator = bilby.gw.waveform_generator.WaveformGenerator(
    duration=4,                      # Duration of the waveform segment (seconds)
    sampling_frequency=2048,         # Sampling rate in Hz
    frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole, # Source model function
    parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters, # Convert parameters to LAL format
    waveform_arguments=waveform_arguments  # Pass waveform-specific arguments
)

# ---------------------------------------------------------------
# STEP 4: Define a function to generate and plot waveform given masses and spins
# ---------------------------------------------------------------
def plot_waveform(mass_1, mass_2, a_1, a_2):
    # Update the injection parameters with values from sliders
    injection_parameters = default_params.copy()
    injection_parameters.update({
        'mass_1': mass_1,
        'mass_2': mass_2,
        'a_1': a_1,
        'a_2': a_2
    })

    # Generate the frequency-domain strain waveform from parameters
    waveform = waveform_generator.frequency_domain_strain(injection_parameters)
    hp = np.array(waveform['plus'])     # Plus polarization component
    hx = np.array(waveform['cross'])    # Cross polarization component
    frequencies = waveform_generator.frequency_array  # Frequency array

    # Plot the absolute value of both polarizations vs frequency
    plt.figure(figsize=(12, 6))
    plt.plot(frequencies, np.abs(hp), label=r"$h_{+}(f)$", linewidth=2)
    plt.plot(frequencies, np.abs(hx), label=r"$h_{\times}(f)$", linestyle='--', linewidth=2)
    plt.xlabel('Frequency [Hz]', fontsize=14)
    plt.ylabel('Strain Amplitude', fontsize=14)
    plt.title('Gravitational Waveform in Frequency Domain', fontsize=16)
    plt.legend(fontsize=12)
    plt.grid(True, linestyle=':')
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------------
# STEP 5: Create interactive sliders for masses and spins
# ---------------------------------------------------------------
mass_1_slider = widgets.FloatSlider(
    value=default_params['mass_1'], min=5, max=150, step=1,
    description=r'Mass 1'
)
mass_2_slider = widgets.FloatSlider(
    value=default_params['mass_2'], min=5, max=150, step=1,
    description=r'Mass 2'
)
a_1_slider = widgets.FloatSlider(
    value=default_params['a_1'], min=0, max=0.99, step=0.01,
    description=r'Spin 1'
)
a_2_slider = widgets.FloatSlider(
    value=default_params['a_2'], min=0, max=0.99, step=0.01,
    description=r'Spin 2'
)

# ---------------------------------------------------------------
# STEP 6: Link sliders with plotting function using interactive_output
# ---------------------------------------------------------------
ui = widgets.VBox([mass_1_slider, mass_2_slider, a_1_slider, a_2_slider]) # User Interface
out = widgets.interactive_output( # Plotting output
    plot_waveform,
    {'mass_1': mass_1_slider, 'mass_2': mass_2_slider, 'a_1': a_1_slider, 'a_2': a_2_slider}
)

# Display the slider UI and output plot together
display(ui, out)


<h3> Quiz Question: <h3>

**Describe the frequency domain waveform signal from above that was generated by the following sections of the code:**

* Injection parameters:

[Fill in Student Answer here]

* Waveform generator:

[Fill in Student Answer here]

* Frequency-domain strain:

[Fill in Student Answer here]

* Graph:

[Fill in Student Answer here]

<details>
<summary> Click to reveal the correct answer and explanation</summary>

* **Injection parameters**: you selected physical properties (masses, spins, distance, orientation) for a binary black hole (BBH) system. These define the “true” source we are simulating.

* **Waveform generator**: the code uses waveform models from LALSimulation to compute the gravitational-wave signal predicted by general relativity for those parameters.

* **Frequency-domain strain**: instead of plotting the signal as strain vs. time, we transformed it into strain vs. frequency. This shows how much signal power exists at each frequency.

* **Graph**: the plot displays the strain amplitudes of the two polarizations ($h_+$ and $h_\times$) as functions of frequency.

</details>

### Important Physical Point:

> If the strain amplitude is larger at low frequencies, that corresponds to the early inspiral phase, when the black holes are still relatively far apart and orbiting more slowly.

> As the system approaches merger, the orbital frequency increases rapidly, and the signal shifts to higher frequencies, where the amplitude typically grows before peaking near merger.

### What Did You Actually Do?

* You changed physical parameters (mass and spin) and observed how those parameters affect the structure of the waveform in frequency space.

In practical terms:

* Increasing the total mass shifts the signal toward lower overall frequencies.

* Changing the mass ratio alters the detailed shape of the spectrum.

* Spin modifies how quickly the system inspirals and merges.

You did not just “make a graph.” You explored how measurable features of a GW signal encode the physical properties of the black holes that produced it.

# Gravitational Waveform Visualization – Time Domain

## Objective: visualize how gravitational waves evolve as two black holes spiral toward each other.

Instructions for Students:

1.	Use the sliders below to adjust:
	- Total Mass (Black Hole 1 + Black Hole 2)
	- Distance (Luminosity distance from the Earth in Megaparsecs)

2.	Observe the changes in the gravitational wave pattern in real time:
	- The waveform’s frequency increases as the black holes get closer—this is called a "chirp".
	- More massive black holes produce stronger gravitational waves.
	- The farther the system is, the weaker the signal received on Earth.


## What to Look For:

* **Frequency increase (“chirp”)**: as the black holes orbit closer together, they move faster. This causes the gravitational-wave frequency to increase over time. The signal’s oscillations become closer together just before merger.

* **Effect of total mass**: increasing the total mass shifts the signal to lower frequencies and typically makes the merger occur more quickly in time. More massive systems also produce larger strain amplitudes at the same distance.

* **Effect of distance**: increasing the distance reduces the strain amplitude. The wave spreads out as it travels, so the signal measured on Earth becomes weaker.

In [ ]:
# -- Simulation parameters --
duration = 4  # seconds
sampling_frequency = 2048  # Hz

# -- Base waveform parameters --
base_params = dict(
    mass_1=35, mass_2=30,
    a_1=0.4, a_2=0.3,
    tilt_1=0, tilt_2=0,
    phi_12=0, phi_jl=0,
    luminosity_distance=500,  # Mpc
    theta_jn=0.4, psi=2.659,
    phase=1.3,
    geocent_time=1126259642.413,
    ra=1.375, dec=-1.2108
)

# -- Waveform generation settings --
waveform_args = dict(
    waveform_approximant='IMRPhenomPv2',
    reference_frequency=50,
    minimum_frequency=20
)

waveform_generator = bilby.gw.WaveformGenerator(
    duration=duration,
    sampling_frequency=sampling_frequency,
    frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
    parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters,
    waveform_arguments=waveform_args
)

# -- Function to update the waveform plot interactively --
def update(total_mass=65, distance=500):
    m1 = m2 = total_mass / 2
    params = base_params.copy()
    params.update({
        "mass_1": m1,
        "mass_2": m2,
        "luminosity_distance": distance
    })

    # Generate frequency domain waveform
    wf_fd = waveform_generator.frequency_domain_strain(params)

    # Convert to time domain
    n_samples = int(duration * sampling_frequency)
    time = np.linspace(0, duration, n_samples)
    hp = np.fft.irfft(wf_fd["plus"], n=n_samples)
    hx = np.fft.irfft(wf_fd["cross"], n=n_samples)

    # Plot
    plt.figure(figsize=(10, 4))
    plt.plot(time, hp, label=r"$h_{+}$ (Plus)")
    plt.plot(time, hx, label=r"$h_{\times}$ (Cross)")

    plt.title(
        rf"Gravitational Waveform Total Mass: {total_mass:.1f} $M_\odot$ — Distance: {distance} Mpc"
    )

    plt.xlabel(r"Time [s]")
    plt.ylabel(r"$\mathrm{strain}$")

    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# -- Sliders for interactive control --
mass_slider = widgets.FloatSlider(value=65, min=20, max=230, step=1, description=r"Total Mass ($M_\odot$)")
distance_slider = widgets.FloatSlider(value=500, min=100, max=5000, step=10, description="Distance (Mpc)")

# -- Display interactive plot --
widgets.interact(update, total_mass=mass_slider, distance=distance_slider)

<h3> Quiz Question: <h3>

Try increasing the total mass to ~225 Solar Masses (around the mass of the heaviest binary seen by LIGO to date, [GW231123](https://www.ligo.caltech.edu/LA/news/ligo20250715)). Then, increase the luminosity distance to 5000 Mpc. **How does the relative strength of the strain change with distance?**

[Fill in Student Answer here].

<details>

<summary> Click to reveal the correct answer and explanation</summary>

Example Student answer:

When I increased the total mass to approximately 225 solar masses (comparable to the system reported in GW231123 by the LIGO–Virgo–KAGRA Collaboration), the waveform became shorter in duration and peaked at a lower frequency. This is expected because higher-mass binaries merge more quickly and emit gravitational waves at lower characteristic frequencies.

After increasing the luminosity distance to 5000 Mpc, the overall amplitude of the strain decreased significantly. The peak strain dropped from roughly $10{^-24}$ to about $10{^-26}$. This demonstrates that gravitational-wave strain scales inversely with luminosity distance:

$h \propto 1 / d_L$

So when the distance increases by a factor of ~ 100, the strain decreases by approximately the same factor (~ 100). The waveform shape remains the same, but its amplitude is uniformly suppressed. This confirms that gravitational waves propagate outward spherically, and their measurable strength weakens proportionally to distance from the source.
</details>

# Gravitational Waveform Visualizations Combined:

In [ ]:
# -----------------------------
# 1. SIMULATED DATA GENERATION
# -----------------------------
def generate_waveforms(amplitude_scale=1.0, frequency_scale=1.0, time_delay=0.0):
    """
    Generate plus and cross gravitational wave polarizations
    with adjustable amplitude, frequency scaling, and time delay.

    The time delay is implemented by shifting the waveform in the time domain,
    using zero-padding to maintain consistent array length.
    """
    duration = 1.0           # Total duration of the waveform in seconds
    sampling_rate = 4096     # Number of samples per second (Hz)
    n_samples = int(duration * sampling_rate)  # Total number of samples
    t = np.linspace(0, duration, n_samples)   # Uniform time array over the duration

    # Chirp frequency evolves linearly from f_start to f_end, scaled by frequency_scale
    f_start = 0.1 * frequency_scale  # Starting frequency in Hz (low to show slow oscillations)
    f_end = 50 * frequency_scale     # Ending frequency in Hz (chirp upward)
    freq = np.linspace(f_start, f_end, n_samples)  # Linearly increasing frequency array

    dt = 1 / sampling_rate  # Time interval between samples

    # Compute instantaneous phase by integrating frequency over time: phase(t) = 2π ∫ f(t) dt
    phase = 2 * np.pi * np.cumsum(freq) * dt

    # Generate base waveforms for the plus and cross polarizations (before any time shift)
    base_h_plus = amplitude_scale * 1e-21 * np.sin(phase)   # Plus polarization waveform
    base_h_cross = amplitude_scale * 1e-21 * np.cos(phase)  # Cross polarization waveform

    # Calculate discrete sample shift corresponding to time_delay in seconds
    sample_shift = int(time_delay * sampling_rate)

    # Helper function to shift waveform in time by integer number of samples
    # Uses zero-padding to avoid wrap-around effects
    def shift_waveform(waveform, shift):
        if shift > 0:
            # Delay waveform: prepend zeros and truncate tail
            shifted = np.concatenate((np.zeros(shift), waveform[:-shift]))
        elif shift < 0:
            # Advance waveform: remove samples from start and append zeros at end
            shifted = np.concatenate((waveform[-shift:], np.zeros(-shift)))
        else:
            # No shift, return original waveform
            shifted = waveform
        return shifted

    # Apply time shift to both polarizations
    h_plus = shift_waveform(base_h_plus, sample_shift)
    h_cross = shift_waveform(base_h_cross, sample_shift)

    # Return time array and shifted waveforms
    return t, h_plus, h_cross


# -----------------------------
# 2. PLOTTING FUNCTION
# -----------------------------
def plot_waveforms(amplitude_scale=1.0, frequency_scale=1.0, time_delay=0.0):
    # Generate waveforms with current slider settings
    t, h_plus, h_cross = generate_waveforms(amplitude_scale, frequency_scale, time_delay)

    # Compute time step for frequency domain analysis
    dt = t[1] - t[0]

    # For demonstration, treat interferometer arms as directly measuring these polarizations
    arm1_response = h_plus
    arm2_response = h_cross

    # Create figure with three vertically stacked subplots
    fig, axs = plt.subplots(3, 1, figsize=(12, 8))
    plt.subplots_adjust(hspace=0.5)  # Space between subplots

    # --- Polarizations in time domain ---
    axs[0].plot(t, h_plus, label=r"Plus polarization ($h_{+}$)", color='blue', lw=2)
    axs[0].plot(t, h_cross, label=r"Cross polarization ($h_{\times}$)", color='orange', lw=2)
    axs[0].set_title("Gravitational Wave Polarizations", fontsize=14)
    axs[0].set_xlabel("Time (s)", fontsize=12)
    axs[0].set_ylabel("Strain", fontsize=12)
    axs[0].grid(True, linestyle=':', alpha=0.7)
    axs[0].set_ylim(-3e-21, 3e-21)  # Fix y-limits for consistent visualization

    # Mark merger time as vertical dashed red line at max |h_plus|
    merger_index = np.argmax(np.abs(h_plus))
    axs[0].axvline(t[merger_index], color='red', linestyle='--', label="Merger", alpha=0.8)
    axs[0].legend(loc='upper right', fontsize=10, framealpha=0.9)

    # --- Interferometer arm responses ---
    axs[1].plot(t, arm1_response, label="Arm 1 Response", color='purple', lw=2)
    axs[1].plot(t, arm2_response, label="Arm 2 Response", color='green', lw=2)
    axs[1].set_title("Interferometer Arm Responses", fontsize=14)
    axs[1].set_xlabel("Time (s)", fontsize=12)
    axs[1].set_ylabel("Strain", fontsize=12)
    axs[1].grid(True, linestyle=':', alpha=0.7)
    axs[1].set_ylim(-3e-21, 3e-21)
    axs[1].legend(loc='upper right', fontsize=10, framealpha=0.9)

    # --- Frequency domain amplitude (FFT) of plus polarization ---
    fft_plus = np.abs(np.fft.rfft(h_plus))  # FFT magnitude
    fft_freq = np.fft.rfftfreq(len(t), dt)  # Corresponding frequencies
    axs[2].plot(fft_freq, fft_plus, color='black', lw=2, label="Plus Polarization FFT")
    axs[2].set_title("Frequency-Domain Signal (Plus Polarization)", fontsize=14)
    axs[2].set_xlabel("Frequency (Hz)", fontsize=12)
    axs[2].set_ylabel("Amplitude", fontsize=12)
    axs[2].grid(True, linestyle=':', alpha=0.7)
    axs[2].legend(loc='upper right', fontsize=10, framealpha=0.9)
    axs[2].set_xlim(0, 500)

    plt.tight_layout()  # Adjust subplots to fit neatly
    plt.show()


# -----------------------------
# 3. INTERACTIVE SLIDERS
# -----------------------------
interact(
    plot_waveforms,
    amplitude_scale=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description="Amplitude"),
    frequency_scale=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description="Frequency"),
    time_delay=FloatSlider(value=0.0, min=-0.5, max=0.5, step=0.01, description="Time Delay (s)")
)


# What are we looking at?

The plot shows a simulated gravitational-wave signal from two black holes spiraling together and merging in three panels: Gravitational Wave Polarizations (top), Interferometer Arm Responses (middle), and the plus polarization plotted in the frequency-domain (bottom).

You are seeing the strain of spacetime as a function of frequency. The oscillations get faster and larger just before the merger — this increasing frequency is a fundamental prediction of general relativity.

The sliders allow you to change physical properties of the system: Amplitude, Frequency, and Time Delay.

The goal is to connect three ideas:

1. The **shape** of the waveform

2. The **physical properties** of the black holes

3. The **signal measured on Earth**

Everything in Gravitational Wave astronomy comes down to interpreting this relationship.

<h3>Quiz Question:<h3>

**Increase the frequency slider while keeping amplitude and time delay the same.**

1. What happens to the spacing between the waves in the top panel?

2. What happens to where the signal appears in the bottom (frequency) plot?

3. If the waves are happening faster, what does that tell you about how quickly the black holes are orbiting each other?

<details>

<summary> Click to reveal the correct answer and explanation</summary>

Example Student Answer:

1. The waves get closer together (more wiggles in the same amount of time).

2. The signal shifts to higher frequency in the bottom plot.

3. Faster waves mean the black holes are orbiting each other faster before they merge.
</details>

# What You Now Understand:

By completing this notebook, you have built on what you learned in Notebook 1 and can now **connect the physics of gravitational waves to real observations and simulations**:

* How Gravitational Wave detectors like LIGO, Virgo, KAGRA, and GEO600 measure tiny ripples in spacetime.

* The difference between binary black hole mergers (e.g., GW150914) and binary neutron star mergers (e.g., GW170817), and why neutron stars merge more slowly.

* How next-generation detectors like the Einstein Telescope (ET) and Cosmic Explorer (CE) will improve detection and measurement of Gravitational Waves.

* What GW strain is and how it encodes information about the masses and orbits of the merging objects.

* How chirp mass affects the speed and shape of the waveform and why it is a key parameter for understanding mergers.

* How to access open GW data from GWOSC and read it into Python for plotting.

* How to generate a theoretical waveform for a merging binary using Bilby, including defining source parameters and waveform arguments.

* How waveform frequency and amplitude change over time, and how scientists visualize these changes to learn about the source.

* How to create an interactive plot to see how changing source masses affects the waveform, similar to professional tools used in GW analysis.

# Coding Skills You Now Have:

In addition to physics, you practiced important coding concepts that scientists use to explore Gravitational Wave data:

* **Reading and handling open data**: loading GW event data files and inspecting their contents.

* **Plotting data in Python**: creating time-domain and frequency-domain graphs using Matplotlib.

* **Defining and using functions**: writing reusable functions for plotting and simulating waveforms.

* **Working with libraries**: using Bilby for waveform generation, along with other scientific libraries (NumPy, Matplotlib, ipywidgets).

* **Interactive coding**: using sliders and dropdowns to create plots where parameters like masses can be changed dynamically.

* **Combining math and code**: translating physical formulas (like chirp mass) into Python calculations.

* **Code commenting and explanation**: writing clear, step-by-step comments so others can understand what your code does.

You now have the tools to explore real GW data, simulate signals, and see how physics and code come together to study cosmic events.


# Where We Go Next:

In this notebook, you learned how to access real GW data, plot it, and generate theoretical waveforms using Python. You also explored how the chirp mass and source parameters shape the signal.

Now we take the next step: extracting scientific information from the data itself.

**In Notebook 3, you will learn how to**:

* Use GW signals to determine the masses and distances of merging objects

* Estimate the properties of black holes and neutron stars from real observations

* Connect the waveform features you simulated to real astrophysical conclusions

Notebook 3 moves from detection and simulation to interpretation and analysis, showing how scientists turn raw signals into knowledge about the universe.

Continue to **[Notebook 3](https://github.com/rlanggin/GW_Explorer_A_Beginners_Guide/blob/main/GW_Explorer_NB3_draft_v2.ipynb)** to start working with parameter estimation and scientific insights.